<a href="https://colab.research.google.com/github/Phani-ISB/Reconciliation_Modules/blob/main/1_Inter_Company_Reconciliation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Importing all the required libraries

In [2]:
# Install RapidFuz library
# RapidFuzz is fast string matching library used for fuzzy comparisions. It is much faster and more efficient (especially for financial datasets)
# Rapidfuzz helps match similar strings wth similarity score (0-100)
!pip install -q rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.7 MB/s eta 0:00:00


In [5]:
import numpy as np
import pandas as pd
from datetime import timedelta
from rapidfuzz import fuzz

# Load the Data

In [20]:
# Datasets from Intercompany reconciliation (One Finance EY portal)
# Dataset name : IC_InputData

df = pd.read_excel("/content/IC_InputData (5).xlsx")

In [21]:
# Check the first few rows of the data
df.head()

,Entity,PartnerEntity,AccountingStandard,Year,Period,TransactionType,TransactionDate,FlexDate1,FlexDate2,KeyIdentifier1,...,DeferredTax,DepreciationForTheYear,OpeningAccumulatedDepreciation,WhetherInClosingStock,PercentOrAmountInClosingStock,Custom1,Custom2,Custom3,GL_Code,GL_Description
0,I053,I052,Algeria GAAP,2026,Year,RR,2023-06-21,2023-06-21,2024-01-09,900000012,...,0,0,0,False,0,NaN,NaN,NaN,1300151,IC Loan Payable
1,I053,I052,Algeria GAAP,2026,Year,RR,2023-06-21,2023-06-21,2024-01-09,900000012,...,0,0,0,False,0,NaN,NaN,NaN,1500001,Intercompany Loans Payable
2,I053,I052,Algeria GAAP,2026,Year,IN,2023-06-21,2023-06-21,2023-11-26,910000007,...,0,0,0,False,0,NaN,NaN,NaN,1300151,IC Loan Payable
3,I053,I052,Algeria GAAP,2026,Year,IN,2023-06-21,2023-06-21,2023-11-26,910000007,...,0,0,0,False,0,NaN,NaN,NaN,1500001,Intercompany Loans Payable
4,I053,I052,Algeria GAAP,2026,Year,IN,2023-04-13,2023-04-13,2024-01-09,910000042,...,0,0,0,False,0,NaN,NaN,NaN,1300151,IC Loan Payable


In [22]:
# Convert Date and Numeric Columns
df['TransactionDate'] =pd.to_datetime(df['TransactionDate'])
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

In [23]:
# Quick summary of entire DataFrame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 328 entries, 0 to 327
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Entity                          328 non-null    object        
 1   PartnerEntity                   328 non-null    object        
 2   AccountingStandard              328 non-null    object        
 3   Year                            328 non-null    int64         
 4   Period                          328 non-null    object        
 5   TransactionType                 328 non-null    object        
 6   TransactionDate                 328 non-null    datetime64[ns]
 7   FlexDate1                       328 non-null    datetime64[ns]
 8   FlexDate2                       328 non-null    datetime64[ns]
 9   KeyIdentifier1                  328 non-null    int64         
 10  KeyIdentifier2                  326 non-null    object        
 11  KeyIde

# Exploratory Data Analysis(EDA)

In [25]:
# Normalize text (Combining narration fields into one and convert to lower case for uniform comparision)
df['Narration'] = (
    df['Narration1'].fillna('').astype(str) + ' ' +
    df['Narration2'].fillna('').astype(str) + ' ' +
    df['Narration3'].fillna('').astype(str)
).str.lower().str.strip()


In [26]:
# Entity Summaries (Total transaction amounts of each entity)
# Quick check before reconciliation, oversall balance of dataset
entity_summary = df.groupby('Entity')['Amount'].sum().reset_index()
print(" \n Entity Summary :" ,entity_summary )
print("\n Total Entities balance check :" , entity_summary['Amount'].sum())

 
 Entity Summary :   Entity       Amount
0   I051    972498.82
1   I052   9209515.41
2   I053    -12872.41
3   I054     42787.00
4   I055   -184879.00
5   I056 -10027049.99
6   I057        -0.05
7   I058  23458999.14
8   I059 -10720459.59
9   I060 -12738538.88

 Total Entities balance check : 0.4499999936670065


In [27]:
#Currency Checks wrt transactions

currency_check = df.groupby(['Entity','PartnerEntity','TransactionCurrency']).size().unstack(fill_value=0)
currency_check

TransactionCurrency   INR
Entity PartnerEntity     
I051   I052             1
I052   I051             1
       I053            70
       I054             9
       I055            10
       I056             8
       I058             2
I053   I052            63
       I055            13
I054   I052             3
I055   I052            10
       I053            13
I056   I052            12
I057   I058            15
I058   I052             6
       I057            18
       I059             9
       I060             8
I059   I058            15
       I060            15
I060   I054             2
       I058             8
       I059            17

In [28]:
# Duplicates Check (Checking identical rows in dataset)
print("Transaction Duplicates:", df.duplicated().sum())

Transaction Duplicates: 0


# Standard Reconciliation

In [29]:
# Initial columns define Reconciliation Status (initial to Unmatched), Type of Rule applied (initial to None) and Reconciliation Group
df['Recon_Status'] = 'Unmatched'
df['Rule_Applied'] = None
df['Recon_ID'] = None

In [30]:
# Matching Functions (Core Logics)
# Allowed Difference in amount while matching (Rounding, Conversion differences, Minor posting mismatches)
AMT_TOLERANCE = 2.0

# Minimum similarity score for narration matching (0-100) [Initial set to moderately similiar i.e 70]
FUZZY_THRESHOLD = 70

#Check if two amounts offset each other witin tolerence
def amount_match(a, b, tol):
    return abs(a + b) <= tol

# Check if two transaction dates are within allowed range
def date_match(d1, d2, days):
    if days is None:
        return True
    return abs((d1 - d2).days) <= days

# Check similarity score between narrations (Fuzzy matching)
def narration_match(n1, n2, fuzzy):
    if fuzzy is None:
        return True
    if not fuzzy:
        return n1 == n2
    return fuzz.partial_ratio(n1, n2) >= FUZZY_THRESHOLD

In [31]:
# Sequential Matching Rules

rules = [
    {"name": "1. Narration Exact + Date Exact + Amount Match", "dt": 0, "fuzzy": False},
    {"name": "2. Narration Fuzzy + Date Exact + Amount Match", "dt": 0, "fuzzy": True},
    {"name": "3. Narration Exact + Date Range + Amount Match", "dt": 5, "fuzzy": False},
    {"name": "4. Narration Fuzzy + Date Range + Amount Match", "dt": 5, "fuzzy": True},
    {"name": "5. Only Date Exact + Amount Match", "dt": 0, "fuzzy": None},
    {"name": "6. Amount Match Only", "dt": None, "fuzzy": None},
    {"name": "7. Within Company Reversal with Amount Match", "dt": None, "fuzzy": None},
    {"name": "8. Multiple matchings + Total Amounts Match (Suggest Manual Check)", "dt": None, "fuzzy": None}
]

In [32]:
# Create unique ID counter for each matched pair
recon_group_id = 1
# Apply rules sequentially
for rule in rules:
    # Filtering unmatched records
    unmatched = df[df['Recon_Status'] == 'Unmatched']
    # Loop through each unmatched transaction
    for i, row in unmatched.iterrows():
     # skip if already matched
        if df.loc[i, 'Recon_Status'] != 'Unmatched':
            continue

        # Cache values
        entity = row['Entity']
        partner = row['PartnerEntity']
        amt = row['Amount']
        date = row['TransactionDate']
        narr = row['Narration']

        # 1. Intercompany candidates (existing swaps entity and partner)
        candidates_1 = unmatched[
            (unmatched['Entity'] == partner) &
            (unmatched['PartnerEntity'] == entity) &
            (unmatched.index != i)
        ]

        # 2. Same Side Candidates (Within Company Reversals)
        candidates_2 = unmatched[
            (unmatched['Entity'] == entity) &
            (unmatched['PartnerEntity'] == partner) &
            (unmatched.index != i)
        ]

        # Combining both candidate sets
        candidates = pd.concat([candidates_1, candidates_2]).drop_duplicates()

        for j, crow in candidates.iterrows():
        # Skipping if already matched
            if df.loc[j, 'Recon_Status'] != 'Unmatched':
                continue
        # Candidate values
            c_amt = crow['Amount']
            c_date = crow['TransactionDate']
            c_narr = crow['Narration']

            # Amount condition
            if round(abs(amt + c_amt), 2) > AMT_TOLERANCE:
                continue

            # Skip date/narration for amount-only rule
            if rule['name'] != "6. Amount Match Only":
            # Date Matching based on rule tolerance (Exact or Range)
                if not date_match(date, c_date, rule['dt']):
                    continue
            # Narration Matching
                if not narration_match(narr, c_narr, rule['fuzzy']):
                    continue

            # Create reconciliation ID for Matches Found
            recon_id = f"REC_{recon_group_id}"
            # Updating both matched rows
            df.loc[[i, j], 'Recon_Status'] = 'Matched'
            df.loc[[i, j], 'Rule_Applied'] = rule['name']
            df.loc[[i, j], 'Recon_ID'] = recon_id
            # Increment of group ID for next match
            recon_group_id += 1
            break

In [33]:
# Rule 8: Multiple matchings (Suggest Manual Check)

multi_match = df[df['Recon_Status'] == 'Unmatched'].copy()

# Create pair (ignore direction)
multi_match['pair'] = multi_match.apply(
    lambda x: tuple(sorted([x['Entity'], x['PartnerEntity']])), axis=1
)

for pair, group in multi_match.groupby('pair'):
    # Apply only if multiple transactions exist
    if len(group) >= 2:
        total = group['Amount'].sum()
        # If near zero OR complex structure → send to manual
        if abs(total) <= AMT_TOLERANCE or len(group) >= 3:
            df.loc[group.index, 'Recon_Status'] = 'Review'
            df.loc[group.index, 'Rule_Applied'] = '8. Multiple matchings (Suggest Manual Check)'
            df.loc[group.index, 'Recon_ID'] = f"REC_{recon_group_id}"

            recon_group_id += 1

In [36]:
# Check and add column in main dataframe as amount differences in the reconciled group
df['Amt_Diff'] = df.groupby('Recon_ID')['Amount'].transform('sum')

In [37]:
# Matched and Unmatched results

matched_df = df[df['Recon_Status'] == 'Matched']
unmatched_df = df[df['Recon_Status'] == 'Unmatched']

In [39]:
matched_df.head()

,Entity,PartnerEntity,AccountingStandard,Year,Period,TransactionType,TransactionDate,FlexDate1,FlexDate2,KeyIdentifier1,...,Custom1,Custom2,Custom3,GL_Code,GL_Description,Narration,Recon_Status,Rule_Applied,Recon_ID,Amt_Diff
0,I053,I052,Algeria GAAP,2026,Year,RR,2023-06-21,2023-06-21,2024-01-09,900000012,...,NaN,NaN,NaN,1300151,IC Loan Payable,i052 40 ici052,Matched,2. Narration Fuzzy + Date Exact + Amount Match,REC_1,0.0
1,I053,I052,Algeria GAAP,2026,Year,RR,2023-06-21,2023-06-21,2024-01-09,900000012,...,NaN,NaN,NaN,1500001,Intercompany Loans Payable,ic loan 19 1300151,Matched,2. Narration Fuzzy + Date Exact + Amount Match,REC_2,0.0
2,I053,I052,Algeria GAAP,2026,Year,IN,2023-06-21,2023-06-21,2023-11-26,910000007,...,NaN,NaN,NaN,1300151,IC Loan Payable,i052 50 ici052,Matched,2. Narration Fuzzy + Date Exact + Amount Match,REC_1,0.0
3,I053,I052,Algeria GAAP,2026,Year,IN,2023-06-21,2023-06-21,2023-11-26,910000007,...,NaN,NaN,NaN,1500001,Intercompany Loans Payable,ic loan 9 1300151,Matched,2. Narration Fuzzy + Date Exact + Amount Match,REC_2,0.0
4,I053,I052,Algeria GAAP,2026,Year,IN,2023-04-13,2023-04-13,2024-01-09,910000042,...,NaN,NaN,NaN,1300151,IC Loan Payable,i052 50 ici052,Matched,2. Narration Fuzzy + Date Exact + Amount Match,REC_3,0.0


In [40]:
# Reconciliation matrix
# Aggregate by Entity → PartnerEntity
matrix = matched_df.groupby(['Entity', 'PartnerEntity'])['Amount'].sum().reset_index()

# Pivot
recon_matrix = matrix.pivot(
    index='Entity',
    columns='PartnerEntity',
    values='Amount'
).fillna(0)

recon_matrix

PartnerEntity,I051,I052,I053,I054,I055,I056,I057,I058,I059,I060
Entity,,,,,,,,,,
I051,0.00,972498.82,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00
I052,-972498.81,0.00,79346.1,-42787.0,102515.0,10027050.0,0.00,0.12,0.00,0.00
I053,0.00,-79346.41,0.0,0.0,82364.0,0.0,0.00,0.00,0.00,0.00
I054,0.00,42787.00,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00
I055,0.00,-102515.00,-82364.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00
I056,0.00,-10027049.99,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00
I057,0.00,0.00,0.0,0.0,0.0,0.0,0.00,-0.05,0.00,0.00
I058,0.00,-0.12,0.0,0.0,0.0,0.0,0.19,0.00,19804343.00,3654656.07
I059,0.00,0.00,0.0,0.0,0.0,0.0,0.00,-19804343.00,0.00,9083883.41


In [41]:
# Summary of Unmatched transactions
matrix = unmatched_df.groupby(['Entity', 'PartnerEntity'])['Amount'].sum().reset_index()

# Pivot
unrecon_matrix = matrix.pivot(
    index='Entity',
    columns='PartnerEntity',
    values='Amount'
).fillna(0)

unrecon_matrix

PartnerEntity
Entity


In [42]:
# Check the unmatched transactions data (if any)
unmatched_df[['Entity','PartnerEntity','Amount','TransactionType','TransactionDate','Narration']]

,Entity,PartnerEntity,Amount,TransactionType,TransactionDate,Narration


In [43]:
# Savng the matching and unmatching transactions to excel files
matched_df.to_excel('matched_transactions.xlsx', index=False)
unmatched_df.to_excel('unmatched_transactions.xlsx', index=False)